In [1]:
import torch
import numpy as np


def get_token(input):
    # english = 'abcdefghijklmnopqrstuvwxyz0123456789'
    english = 'abcdefghijklmnopqrstuvwxyz'
    output = []
    buffer = ''
    for s in input:
        if s in english or s in english.upper():
            buffer += s
        else:
            if buffer: output.append(buffer)
            buffer = ''
            output.append(s)
    if buffer: output.append(buffer)
    return output


from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained('./output/checkpoint-2000')
print(model)
print(model.config.id2label)

from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained('bert-base-chinese')

Loading weights:   0%|          | 0/25 [00:00<?, ?it/s]

AlbertForTokenClassification(
  (albert): AlbertModel(
    (embeddings): AlbertEmbeddings(
      (word_embeddings): Embedding(21128, 128, padding_idx=0)
      (position_embeddings): Embedding(512, 128)
      (token_type_embeddings): Embedding(2, 128)
      (LayerNorm): LayerNorm((128,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0, inplace=False)
    )
    (encoder): AlbertTransformer(
      (embedding_hidden_mapping_in): Linear(in_features=128, out_features=768, bias=True)
      (albert_layer_groups): ModuleList(
        (0): AlbertLayerGroup(
          (albert_layers): ModuleList(
            (0): AlbertLayer(
              (full_layer_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
              (attention): AlbertAttention(
                (attention_dropout): Dropout(p=0, inplace=False)
                (output_dropout): Dropout(p=0, inplace=False)
                (query): Linear(in_features=768, out_features=768, bias=True)
                (k

In [2]:
input_str = '2009年高考在北京的报名费是2009元'
input_char = get_token(input_str)
input_char

['2',
 '0',
 '0',
 '9',
 '年',
 '高',
 '考',
 '在',
 '北',
 '京',
 '的',
 '报',
 '名',
 '费',
 '是',
 '2',
 '0',
 '0',
 '9',
 '元']

In [3]:
input_tensor = tokenizer(input_char, is_split_into_words=True, padding=True, truncation=True,
                            return_offsets_mapping=True, max_length=512, return_tensors="pt")
input_tokens = input_tensor.tokens()
offsets = input_tensor["offset_mapping"]
ignore_mask = offsets[0, :, 1] == 0

input_tensor.pop("offset_mapping")  # 不剔除的话会报错
outputs = model(**input_tensor)
probabilities = torch.nn.functional.softmax(outputs.logits, dim=-1)[0].tolist()
predictions = outputs.logits.argmax(dim=-1)[0].tolist()
predictions

[3, 4, 1, 1, 1, 1, 0, 0, 0, 5, 3, 0, 0, 0, 0, 0, 5, 3, 3, 3, 3, 0]

In [7]:
for pred in predictions:
    print(model.config.id2label[pred], end=",")
# input_str = '2009年高考在北京的报名费是2009元'

I-place,B-year,I-year,I-year,I-year,I-year,O,O,O,B-place,I-place,O,O,O,O,O,B-place,I-place,I-place,I-place,I-place,O,

In [ ]:
results = []
tokens = input_tensor.tokens()
idx = 0
while idx < len(predictions):
    if ignore_mask[idx]:
        idx += 1
        continue
    pred = predictions[idx]
    label = model.config.id2label[pred]
    if label != "O":
        # 不加B-或者I-
        label = label[2:]
        start = idx
        end = start + 1
        # 获取所有token I-label
        all_scores = []
        all_scores.append(probabilities[start][predictions[start]])
        while (
            end < len(predictions)
            and model.config.id2label[predictions[end]] == f"I-{label}"
        ):
            all_scores.append(probabilities[end][predictions[end]])
            end += 1
            idx += 1
        # 得到是他们平均的
        score = np.mean(all_scores).item()
        word = input_tokens[start:end]
        results.append(
            {
                "entity_group": label,
                "score": score,
                "word": word,
                "start": start,
                "end": end,
            }
        )
    idx += 1

In [9]:
for i in range(len(results)):
    print(results[i])

{'entity_group': 'year', 'score': 0.9978174090385437, 'word': ['2', '0', '0', '9', '年'], 'start': 1, 'end': 6}
{'entity_group': 'place', 'score': 0.9886337518692017, 'word': ['北', '京'], 'start': 9, 'end': 11}
{'entity_group': 'place', 'score': 0.33192232847213743, 'word': ['2', '0', '0', '9', '元'], 'start': 16, 'end': 21}


In [13]:
probabilities[16:22]

[[0.14251160621643066,
  0.04223678633570671,
  0.1958564966917038,
  0.09257909655570984,
  0.03569323197007179,
  0.3760782778263092,
  0.11504454165697098],
 [0.0646522045135498,
  0.12379178404808044,
  0.054375823587179184,
  0.34088510274887085,
  0.09717822819948196,
  0.18152429163455963,
  0.1375926434993744],
 [0.068743497133255,
  0.12494496256113052,
  0.06762640923261642,
  0.3214836120605469,
  0.11403046548366547,
  0.17186208069324493,
  0.131308913230896],
 [0.06670518964529037,
  0.1428937464952469,
  0.08043825626373291,
  0.2894829511642456,
  0.11498983204364777,
  0.1740054041147232,
  0.13148462772369385],
 [0.09758038818836212,
  0.1248902827501297,
  0.08869801461696625,
  0.3316816985607147,
  0.06983733177185059,
  0.17157426476478577,
  0.11573808640241623],
 [0.9997847676277161,
  2.9177890610299073e-05,
  3.3737567719072104e-05,
  4.0864371840143576e-05,
  3.40403312293347e-05,
  3.0823342967778444e-05,
  4.651514609577134e-05]]